In [1]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 58.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import pathlib
from transformers import VivitImageProcessor, VivitForVideoClassification
from PIL import Image
from torch.utils.data import Dataset
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
import numpy as np
import random
import av
from importlib.resources import path
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import time

In [3]:
dataset_root_path = pathlib.Path("/kaggle/input/datasets/beibarszhenis/wlasl-300/wlasl300")

video_count_train = len(list(dataset_root_path.glob("train/*/*.mp4")))
video_count_val = len(list(dataset_root_path.glob("val/*/*.mp4")))
video_count_test = len(list(dataset_root_path.glob("test/*/*.mp4")))
video_total = video_count_train + video_count_val + video_count_test
print(f"Total videos: {video_total}")

all_video_file_paths = (
    list(dataset_root_path.glob("train/*/*.mp4"))
    + list(dataset_root_path.glob("val/*/*.mp4"))
    + list(dataset_root_path.glob("test/*/*.mp4"))
)
all_video_file_paths[:5]

class_labels = sorted({path.parent.name for path in all_video_file_paths})

label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"Unique classes ({len(class_labels)}): {class_labels}")

Total videos: 5118
Unique classes (300): ['about', 'accident', 'africa', 'again', 'all', 'always', 'animal', 'apple', 'approve', 'argue', 'arrive', 'baby', 'back', 'backpack', 'bad', 'bake', 'balance', 'ball', 'banana', 'bar', 'basketball', 'bath', 'bathroom', 'beard', 'because', 'bed', 'before', 'behind', 'bird', 'birthday', 'black', 'blanket', 'blue', 'book', 'bowling', 'boy', 'bring', 'brother', 'brown', 'business', 'but', 'buy', 'call', 'can', 'candy', 'careful', 'cat', 'catch', 'center', 'cereal', 'chair', 'champion', 'change', 'chat', 'cheat', 'check', 'cheese', 'children', 'christmas', 'city', 'class', 'clock', 'close', 'clothes', 'coffee', 'cold', 'college', 'color', 'computer', 'convince', 'cook', 'cool', 'copy', 'corn', 'cough', 'country', 'cousin', 'cow', 'crash', 'crazy', 'cry', 'cute', 'dance', 'dark', 'daughter', 'day', 'deaf', 'decide', 'delay', 'delicious', 'different', 'disappear', 'discuss', 'divorce', 'doctor', 'dog', 'door', 'draw', 'dress', 'drink', 'drive', 'drop'

In [4]:
image_processor = VivitImageProcessor.from_pretrained("Shawon16/ViViT_wlasl_100_200ep_coR_")
model = VivitForVideoClassification.from_pretrained(
    "Shawon16/ViViT_wlasl_100_200ep_coR_",
    label2id = label2id,
    id2label = id2label,
    ignore_mismatched_sizes = True,
)

DEVICE = torch.device("cuda")
model.to(DEVICE)
print(next(model.parameters()).device)

preprocessor_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

VivitForVideoClassification LOAD REPORT from: Shawon16/ViViT_wlasl_100_200ep_coR_
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100, 768]) vs model:torch.Size([300, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([100]) vs model:torch.Size([300])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


cuda:0


In [5]:
NUM_FRAMES = 32 
NUM_CLIPS = 2

class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, image_processor, num_frames, train=True):
        self.video_paths = video_paths
        self.labels = labels
        self.image_processor = image_processor
        self.num_frames = num_frames
        self.train = train

    def __len__(self):
        return len(self.video_paths)

    def _load_video(self, path):
        container = av.open(str(path))
        frames = []
        for frame in container.decode(video=0):
            img = Image.fromarray(frame.to_rgb().to_ndarray())
            img = img.resize((224, 224))
            frames.append(np.array(img))
        container.close()
        return frames

    def _sample_frames_uniform(self, frames):
        idx = np.linspace(0, len(frames) - 1, self.num_frames).astype(int)
        return [frames[i] for i in idx]

    def _get_inference_clips(self, frames):
        total_needed = NUM_CLIPS * self.num_frames 

        if len(frames) < total_needed:
            reps = total_needed // len(frames) + 1
            frames = (frames * reps)[:total_needed]

        clip0 = frames[0 : self.num_frames]
        clip1 = frames[self.num_frames : 2 * self.num_frames]
        return [clip0, clip1]

    def __getitem__(self, idx):
        frames = self._load_video(self.video_paths[idx])
        label = torch.tensor(self.labels[idx])

        if self.train:
            sampled = self._sample_frames_uniform(frames)
            inputs = self.image_processor(sampled, return_tensors="pt")
            pixel_values = inputs["pixel_values"].squeeze(0)
        else:
            clips = self._get_inference_clips(frames)
            clip_tensors = []
            for clip in clips:
                inp = self.image_processor(clip, return_tensors="pt")
                clip_tensors.append(inp["pixel_values"].squeeze(0))
            pixel_values = torch.stack(clip_tensors, dim=0)

        return {"pixel_values": pixel_values, "labels": label}

In [6]:
train_paths = list((dataset_root_path / "train").rglob("*.mp4"))
val_paths = list((dataset_root_path / "val").rglob("*.mp4"))
test_paths = list((dataset_root_path / "test").rglob("*.mp4"))

train_labels = [label2id[p.parent.name] for p in train_paths]
val_labels = [label2id[p.parent.name] for p in val_paths]
test_labels = [label2id[p.parent.name] for p in test_paths]

train_dataset = VideoDataset(train_paths, train_labels, image_processor, NUM_FRAMES, train=True)
val_dataset = VideoDataset(val_paths,   val_labels,   image_processor, NUM_FRAMES, train=False)
test_dataset = VideoDataset(test_paths,  test_labels,  image_processor, NUM_FRAMES, train=False)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=2)
test_loader = DataLoader(test_dataset,  batch_size=2)

In [7]:
EPOCHS = 15
LR = 1e-3
WARMUP_EPOCHS = 5
MOMENTUM = 0.9 
WEIGHT_DECAY = 0.01
GRAD_ACCUM_STEPS = 8
PATIENCE = 5
MIN_IMPR = 0.001

SAVE_PATH = "/kaggle/working/vivit300-finetuned.pth"

In [8]:
backbone_params = [p for n, p in model.named_parameters() if "classifier" not in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.SGD(
    [
        {"params": backbone_params, "lr": LR * 0.1},
        {"params": head_params, "lr": LR}, 
    ],
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

warmup  = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine  = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

scaler = torch.amp.GradScaler('cuda')

best_val_loss = float('inf')
patience_counter = 0

start = time.time()
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}  (lr={scheduler.get_last_lr()[0]:.2e})")
    print("-" * 30)
    model.train()
    train_loss = 0
    optimizer.zero_grad()

    train_preds = []
    train_labels = []

    for step, batch in enumerate(tqdm(train_loader)):
        inputs = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            logits = outputs.logits
            loss = criterion(logits, labels) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * GRAD_ACCUM_STEPS

        preds = torch.argmax(logits, dim=1)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(labels.detach().cpu().numpy())

    train_acc = accuracy_score(train_labels, train_preds)
    avg_train_loss = train_loss / len(train_loader)
    
    model.eval()
    val_preds = []
    val_labels = []
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader):
            pv = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            B = pv.shape[0]

            pv_flat = pv.view(B * NUM_CLIPS, *pv.shape[2:])

            with torch.amp.autocast('cuda'):
                logits_flat = model(pv_flat).logits 

            logits = logits_flat.view(B, NUM_CLIPS, -1).mean(dim=1)
            loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    avg_val_loss = val_loss / len(val_loader)

    scheduler.step()

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Val Loss  : {avg_val_loss:.4f}")
    print(f"Val Acc   : {val_acc:.4f}")

    if avg_val_loss < best_val_loss - MIN_IMPR:
        print("Validation loss improved.")
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_loss": best_val_loss,
            "patience_counter": patience_counter,
        }, SAVE_PATH)
    else:
        patience_counter += 1
        print(f"No significant improvement. Patience {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print("Training complete.")
print(f"Training time: {time.time() - start:.4f}")

model.load_state_dict(torch.load(SAVE_PATH)["model_state_dict"])


Epoch 1/15  (lr=1.00e-05)
------------------------------


100%|██████████| 451/451 [06:58<00:00,  1.08it/s]


Train Loss: 5.8336
Train Acc : 0.0031
Val Loss  : 5.8336
Val Acc   : 0.0022
Validation loss improved.

Epoch 2/15  (lr=2.80e-05)
------------------------------


100%|██████████| 451/451 [06:53<00:00,  1.09it/s]


Train Loss: 5.8154
Train Acc : 0.0034
Val Loss  : 5.8130
Val Acc   : 0.0022
Validation loss improved.

Epoch 3/15  (lr=4.60e-05)
------------------------------


...

100%|██████████| 451/451 [06:57<00:00,  1.08it/s]


Train Loss: 5.1392
Train Acc : 0.1333
Val Loss  : 5.3563
Val Acc   : 0.0710
Validation loss improved.

Epoch 15/15  (lr=2.45e-06)
------------------------------


100%|██████████| 451/451 [06:57<00:00,  1.08it/s]


Train Loss: 5.1325
Train Acc : 0.1355
Val Loss  : 5.3549
Val Acc   : 0.0721
Validation loss improved.
Training complete.
Training time: 33278.2368


<All keys matched successfully>

In [9]:
model.eval()
test_preds = []
test_labels_list = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        pv = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        B = pv.shape[0]

        pv_flat = pv.view(B * NUM_CLIPS, *pv.shape[2:])

        with torch.amp.autocast('cuda'):
            logits_flat = model(pv_flat).logits 

        logits = logits_flat.view(B, NUM_CLIPS, -1).mean(dim=1)

        preds = torch.argmax(logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels_list.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_labels_list, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

100%|██████████| 334/334 [05:17<00:00,  1.05it/s]

Test Accuracy: 0.0704
